# 01. Data Preprocessing

Tạo `df_feature.csv`, `df_target.csv` từ tập train và `test_df_feature.csv` từ tập test. Transformer chỉ fit trên train rồi dùng lại cho test.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src import config as C
from src.preprocessing import load_raw, missing_summary, fit_transformers, transform

pd.set_option("display.max_columns", 80)

In [2]:
train = load_raw(C.TRAIN_RAW)
test = load_raw(C.TEST_RAW)
train.shape, test.shape

((136992, 44), (34080, 44))

In [3]:
train.head()

,customer_id,treatment,cash_sign_30d,card_sign_30d,min_sign_vol,avg_sign_vol,cnt_month_from_last_appl,cli_age,cli_job,cli_marital_status,cli_contact_region_level,cli_education,cli_emp_period,last_offer_limit,last_process_channel,f_approve_last_process,cnt_connected_tls,cnt_fail_tls_3m,f_appointment_3m,cnt_total_payment,risk_grp,cli_contact_region_population_grp,ratio_monthly_amt_l6m_vs_n6m,worst_credit_status,cnt_month_status1,ratio_vol_hcvn_cash_appl_12m,f_ever_default,ratio_conn_tls_weekend_12m,ratio_conn_tls_endmonth_12m,cnt_bank_inst_appl_12m,cnt_fi_inst_appl_12m,cnt_report_bank_6m,cnt_report_institution_12m,cnt_lender_has2pay_last_24m,max_cntr_len_fi_12m,amt_terminated_loan_12m,cnt_terminated_loan_12m,cnt_month_over3_credit_card_6m,cnt_noinsurance_loan_24m,cnt_earlyterminated_cash,cnt_earlyterminted_pos,cnt_inbound_tls_12m,cnt_month_from_first_transaction,cnt_month_from_last_transaction
0,11,cash,0,0,4141000.0,3986000.0,141,25,ENGINEER,Single,South,PTTH,167.0,20020000.0,NaN,NaN,13,NaN,NaN,18.0,E,URBAN,NaN,NaN,NaN,NaN,0.0,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,0,NaN,NaN,NaN
1,19,card,0,0,2290000.0,3140000.0,12,37,WORKER,Married,Central,PTTH,56.0,31077000.0,D,1.0,4,12.0,0.0,15.0,D,RURAL,0.6,1.0,16.0,NaN,0.0,0.25,0.00,0.0,2.0,0.0,2.0,2.0,183.0,2312000.0,1.0,0.0,0.0,0,0,0.0,NaN,NaN
2,48,card,0,0,1604000.0,2943000.0,6,34,FARMER,Married,South,PTCS,43.0,13867000.0,D,0.0,18,NaN,NaN,30.0,E,SUB_URBAN,NaN,6.0,25.0,NaN,0.0,0.00,0.00,0.0,0.0,0.0,0.0,1.0,NaN,0.0,0.0,0.0,0.0,0,0,NaN,NaN,NaN
3,59,card,0,0,5090000.0,7839000.0,9,29,FARMER,Single,South,PTCS,67.0,83747000.0,D,1.0,40,13.0,0.0,56.0,C,RURAL,NaN,1.0,20.0,NaN,1.0,0.27,0.47,0.0,1.0,0.0,1.0,1.0,245.0,0.0,0.0,0.0,0.0,0,0,0.0,9.0,9.0
4,60,cash,0,0,7617000.0,9165000.0,71,30,SERVICES,Married,Central,PTTH,86.0,51956000.0,G,1.0,50,NaN,NaN,22.0,C,SUB_URBAN,NaN,1.0,11.0,NaN,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0,0,NaN,NaN,NaN


## 1. Missing value

| Missing % | Mức độ |
|---|---|
| 0% | no |
| (0%, 5%] | very_few |
| (5%, 20%] | few |
| (20%, 50%] | medium |
| (50%, 80%] | many |
| > 80% | very_many |

In [4]:
schema = missing_summary(train)
schema.sort_values("null_pct", ascending=False)

,dtype,non_null,null,null_pct,unique,missing_level
cnt_month_from_last_transaction,float64,8773,128219,93.596,86,very_many
cnt_month_from_first_transaction,float64,8773,128219,93.596,97,very_many
ratio_vol_hcvn_cash_appl_12m,float64,14426,122566,89.469,99,very_many
ratio_monthly_amt_l6m_vs_n6m,float64,23901,113091,82.553,1261,very_many
max_cntr_len_fi_12m,float64,35518,101474,74.073,509,many
f_appointment_3m,float64,60694,76298,55.695,2,many
cnt_fail_tls_3m,float64,82660,54332,39.661,62,medium
cnt_inbound_tls_12m,float64,82660,54332,39.661,6,medium
cnt_lender_has2pay_last_24m,float64,96483,40509,29.570,10,medium
last_process_channel,str,96864,40128,29.292,9,medium


In [5]:
schema.reset_index(names="column").groupby("missing_level", observed=False)["column"].agg(["count", list])

,count,list
missing_level,,
no,10,"[customer_id, treatment, cash_sign_30d, card_s..."
very_few,11,"[min_sign_vol, avg_sign_vol, cli_contact_regio..."
few,3,"[cli_marital_status, cli_emp_period, f_ever_de..."
medium,14,"[last_process_channel, f_approve_last_process,..."
many,2,"[f_appointment_3m, max_cntr_len_fi_12m]"
very_many,4,"[ratio_monthly_amt_l6m_vs_n6m, ratio_vol_hcvn_..."


## 2. Nhóm cột

| Nhóm | Xử lý |
|---|---|
| Số liên tục | `SimpleImputer(median)`, `MinMaxScaler` |
| Số rời rạc, flag | `SimpleImputer(most_frequent)`, `MinMaxScaler` |
| Nominal | `SimpleImputer(most_frequent)`, `OneHotEncoder(drop="first")` |
| Ordinal (`cli_education`) | `SimpleImputer(most_frequent)`, `OrdinalEncoder`, `MinMaxScaler` |
| Missing trên 50% và `customer_id` | Bỏ |

In [6]:
pd.Series({
    "high_missing (drop)": len(C.HIGH_MISSING_COLS),
    "numeric continuous": len(C.NUM_CONT_COLS),
    "numeric discrete": len(C.NUM_CAT_COLS),
    "nominal": len(C.NOMINAL_COLS),
    "ordinal": len(C.ORDINAL_COLS),
    "target": len(C.TARGETS),
})

high_missing (drop)     6
numeric continuous     15
numeric discrete       12
nominal                 7
ordinal                 1
target                  2
dtype: int64

In [7]:
for col in C.NOMINAL_COLS + C.ORDINAL_COLS:
    print(col, train[col].unique().tolist())

treatment ['cash', 'card']
cli_job ['ENGINEER', 'WORKER', 'FARMER', 'SERVICES', 'XNA', 'SALESMAN', 'OTHER', 'ADMINISTRATIVE']
cli_marital_status ['Single', 'Married', 'XNA', nan, 'Divorced', 'Widowed', 'Separated', 'Living with partner']
cli_contact_region_level ['South', 'Central', 'North', nan]
last_process_channel [nan, 'D', 'G', 'H', 'I', 'C', 'E', 'B', 'F', 'J']
risk_grp ['E', 'D', 'C', 'B', 'A', nan, 'F', 'G', 'H', 'I']
cli_contact_region_population_grp ['URBAN', 'RURAL', 'SUB_URBAN', 'KEY_URBAN', nan]
cli_education ['PTTH', 'PTCS', 'ELEMENTARY_SCHOOL', 'BACHELORS', 'MASTERS', nan]


## 3. Fit trên train, transform train và test

In [8]:
train.to_csv(C.TRAIN_CLEANED, index=False)

train_model = train.drop(columns=C.HIGH_MISSING_COLS + [C.ID_COL])
test_model = test.drop(columns=C.HIGH_MISSING_COLS + [C.ID_COL] + C.TARGETS)

transformers = fit_transformers(train_model)
df_feature = transform(train_model, transformers)
df_target = train_model[C.TARGETS]
test_df_feature = transform(test_model, transformers)

df_feature.shape, df_target.shape, test_df_feature.shape

((136992, 63), (136992, 2), (34080, 63))

In [9]:
df_feature.describe().T[["min", "max"]].agg(["min", "max"])

,min,max
min,0.0,1.0
max,0.0,1.0


In [10]:
df_target.mean().rename("positive_rate")

cash_sign_30d    0.059186
card_sign_30d    0.027111
Name: positive_rate, dtype: float64

In [11]:
import joblib

df_feature.to_csv(C.TRAIN_FEATURE, index=False)
df_target.to_csv(C.TRAIN_TARGET, index=False)
test_df_feature.to_csv(C.TEST_FEATURE, index=False)
joblib.dump(transformers, C.TRANSFORMERS)

['/home/user/DAZONE-ROUND-3---HOME-CREDIT---UPLIFT-MODEL/data/processed/transformers.joblib']